# Chapters 08–10 · Git & GitHub: hands-on

Companion to `notes/08_…`, `09_…` and `10_…` (video 02:08:22 – 02:25:53).

Everything the instructor does with Git and GitHub, run for real, **offline**:

* A **bare repository** (`github-learning-test.git`) plays the role of GitHub. Pushing to it and pulling from it works exactly like with github.com.
* Two clones, **`dev1`** and **`dev2`**, play two developers on different machines.
* Git settings come from a sandbox config file, so **your real `~/.gitconfig` is never read or changed**.

To do it on the real GitHub, create the repo on github.com (Chapter 08) and use its URL instead of the local path. The commands are the same.

In [1]:
import os, shutil

SANDBOX = os.path.abspath("git_sandbox")
shutil.rmtree(SANDBOX, ignore_errors=True)
os.makedirs(SANDBOX)
os.chdir(SANDBOX)

# isolated identity/config for this notebook only
cfg = os.path.join(SANDBOX, "sandbox.gitconfig")
with open(cfg, "w") as f:
    f.write("""[user]
    name = Learner
    email = learner@example.com
[init]
    defaultBranch = main
[pull]
    rebase = false
[advice]
    detachedHead = false
[color]
    ui = never
""")
os.environ["GIT_CONFIG_GLOBAL"] = cfg     # use this instead of ~/.gitconfig
os.environ["GIT_CONFIG_NOSYSTEM"] = "1"
os.environ["GIT_PAGER"] = "cat"
for k in ("CLICOLOR", "CLICOLOR_FORCE", "LS_COLORS"):
    os.environ.pop(k, None)  # plain (uncoloured) ls output
print("sandbox:", SANDBOX)

sandbox: /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/git_sandbox


---
# Chapter 08 · Create the remote repo and clone it

## 8.1 · "Create repository" on GitHub
On github.com you click **New → name → Public → ☑ README → .gitignore: Python → License: MIT → Create**.
GitHub then makes an **initial commit** containing those three files. We do the same by hand:

In [2]:
%%bash
git init --bare -q github-learning-test.git          # the empty "GitHub" repo

# what GitHub's form does for you: an initial commit with README, .gitignore, LICENSE
git clone -q github-learning-test.git _github_web 2>/dev/null
cd _github_web
printf '# github-learning-test\nLearning Git & GitHub for MLOps.\n' > README.md
printf '__pycache__/\n*.py[cod]\n.venv/\nvenv/\n.env\n.ipynb_checkpoints\n*.log\n' > .gitignore
printf 'MIT License\n\nCopyright (c) 2026 Learner\n\n(Full MIT text: https://opensource.org/license/mit)\n' > LICENSE
git add . && git commit -q -m "Initial commit"
git push -q origin main
cd .. && rm -rf _github_web
echo "remote repo ready"

remote repo ready


## 8.2 · `git clone`  (video 02:11)
On GitHub: **Code → HTTPS → copy**, then `git clone <url>`. Here the "URL" is the local path.

In [3]:
%%bash
git clone github-learning-test.git dev1
cd dev1
echo "--- ls -a (note the hidden .git folder) ---"; ls -a
echo "--- inside .git ---"; ls .git
echo "--- the remote called origin ---"; git remote -v

Cloning into 'dev1'...


done.


--- ls -a (note the hidden .git folder) ---


.
..
.git
.gitignore
LICENSE
README.md


--- inside .git ---


config
description
HEAD
hooks
index
info
logs
objects
packed-refs
refs


--- the remote called origin ---


origin	/Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/git_sandbox/github-learni

ng-test.git (fetch)
origin	/Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/git_s

andbox/github-learning-test.git (push)


## 8.3 · What `.gitignore` does
`git check-ignore -v` tells you which rule (if any) makes Git skip a file.

In [4]:
%%bash
cd dev1
mkdir -p src/__pycache__ && touch app.py .env server.log src/__pycache__/utils.cpython-313.pyc
for f in app.py .env server.log src/__pycache__/utils.cpython-313.pyc; do
  git check-ignore -q "$f" && echo "IGNORED  $f  ← $(git check-ignore -v "$f" | cut -f1)" || echo "tracked  $f"
done
git status --short         # only app.py shows up as a new file
rm -rf app.py .env server.log src

tracked  app.py


IGNORED  .env  ← .gitignore:5:.env


IGNORED  server.log  ← .gitignore:7:*.log


IGNORED  src/__pycache__/utils.cpython-313.pyc  ← .gitignore:1:__pycache__/


?? app.py


---
# Chapter 09 · status → add → commit → push → pull

## 9.1 · A new file shows up as *untracked*  (video 02:14)

In [5]:
%%bash
cd dev1
printf 'print("feature 1")\n' > test.py
git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file

>..." to include in what will be committed)
	test.py

nothing added to commit but untracked files pr

esent (use "git add" to track)


## 9.2 · Stage just one file  (video 02:15)
With two new files, `git add test.py` stages only `test.py`, and `test2.py` stays untracked.

In [6]:
%%bash
cd dev1
printf 'print("bappy")\n' > test2.py
git add test.py
git status
echo "--- short form: A = staged new file, ?? = untracked ---"
git status --short

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git r

estore --staged <file>..." to unstage)
	new file:   test.py

Untracked files:
  (use "git add <file>

..." to include in what will be committed)
	test2.py



--- short form: A = staged new file, ?? = untracked ---


A  test.py
?? test2.py


## 9.3 · Commit, then push to `origin main`  (video 02:16 – 02:17)

In [7]:
%%bash
cd dev1
git commit -m "test.py file added"
echo "--- status: test.py is gone from the list (it's committed and tracked now) ---"
git status --short
git push origin main 2>&1
git log --oneline

[main 77484a4] test.py file added
 1 file changed, 1 insertion(+)
 create mode 100644 test.py


--- status: test.py is gone from the list (it's committed and tracked now) ---


?? test2.py


To /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/git_sandbox/github-learning-t

est.git
   c7b0401..77484a4  main -> main


77484a4 test.py file added


c7b0401 Initial commit


## 9.4 · A second developer pulls, adds feature 2 and pushes  (video 02:18)

In [8]:
%%bash
git clone -q github-learning-test.git dev2
cd dev2
echo "--- dev2 received ---"; cat test.py
printf 'print("feature 2")\n' >> test.py
git status --short                   # M = modified
git add test.py
git commit -q -m "test.py file updated"
git push -q origin main && echo "dev2 pushed feature 2"

--- dev2 received ---


print("feature 1")


 M test.py


dev2 pushed feature 2


In [9]:
%%bash
cd dev1
git pull 2>&1 | tail -3
echo "--- dev1 now has ---"; cat test.py

Fast-forward
 test.py | 1 +
 1 file changed, 1 insertion(+)


--- dev1 now has ---


print("feature 1")
print("feature 2")


## 9.5 · Why you pull before you push (beyond the video)
If someone pushed while you were working, **your push is rejected** until you pull their commits.

In [10]:
%%bash
cd dev2 && printf 'x = 1\n' > utils.py && git add . && git commit -q -m "add utils" && git push -q origin main && cd ..
cd dev1 && printf '# notes\n' > NOTES.md && git add . && git commit -q -m "add notes"
git push origin main 2>&1 | head -3 || true
echo "↑ rejected: the remote has a commit dev1 doesn't have"
git pull --no-edit 2>&1 | tail -2        # different files, so Git merges automatically
git push -q origin main && echo "push OK after pulling"

To /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/git_sandbox/github-learning-t

est.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to '/Users/h

emanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/git_sandbox/github-learning-test.git'


↑ rejected: the remote has a commit dev1 doesn't have


 1 file changed, 1 insertion(+)
 create mode 100644 utils.py


push OK after pulling


## 9.6 · Going back to a previous version  (video 02:19)

The video suggests using `git pull` to get old code back. **That doesn't work:** `pull` only brings the *latest* commits.
The real tools are `git log`, `git show`, `git restore --source` and `git revert`:

In [11]:
%%bash
cd dev1
git log --oneline
OLD=$(git log --format=%h --grep="test.py file added" -n1)
echo "--- test.py as it was in $OLD ---"
git show "$OLD":test.py

530417d Merge branch 'main' of /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/g

it_sandbox/github-learning-test


d32222e add notes
b416f04 add utils
51051d1 test.py file updated
77484a4 test.py file added
c7b0401 

Initial commit


--- test.py as it was in 77484a4 ---


print("feature 1")


In [12]:
%%bash
cd dev1
OLD=$(git log --format=%h --grep="test.py file added" -n1)
git restore --source "$OLD" test.py        # bring the old file back into the working folder
echo "--- working copy now ---"; cat test.py
git diff --stat
git restore test.py                        # undo: back to the latest version
echo "--- restored latest ---"; cat test.py

--- working copy now ---


print("feature 1")


 test.py | 1 -
 1 file changed, 1 deletion(-)


--- restored latest ---


print("feature 1")
print("feature 2")


In [13]:
%%bash
cd dev1
printf 'print(1/0)  # bug!\n' >> test.py
git commit -q -am "buggy change" && git push -q origin main
BAD=$(git log --format=%h -n1)
git revert --no-edit "$BAD" 2>&1 | head -2   # a NEW commit that undoes the bad one (safe after pushing)
git push -q origin main
cat test.py
git log --oneline -4

[main e8c53e0] Revert "buggy change"
 Date: Wed Sep 16 20:18:53 2026 -0400


print("feature 1")
print("feature 2")


e8c53e0 Revert "buggy change"


3b72f8d buggy change
530417d Merge branch 'main' of /Users/hemanthreddy/Desktop/data_science/MLOps/p

rojects/ch08_10_git/git_sandbox/github-learning-test
d32222e add notes


## 9.7 · Commit everything with `git add .`  (video 02:20)

In [14]:
%%bash
cd dev1
touch test3.py
git status --short
git add .
git commit -q -m "files added"
git push -q origin main
git log --oneline -3

?? test3.py


2491c7b files added


e8c53e0 Revert "buggy change"
3b72f8d buggy change


---
# Chapter 10 · Branches

## 10.1 · List branches, then create and switch  (video 02:22)

In [15]:
%%bash
cd dev1
git branch
git checkout -b bappy          # newer equivalent: git switch -c bappy
git branch

* main


Switched to a new branch 'bappy'


* bappy
  main


## 10.2 · Commit on the branch and push the branch  (video 02:23)

In [16]:
%%bash
cd dev1
printf 'print("hello world")\nprint("this is new")\n' > test3.py
git add . && git commit -q -m "new code added"
git push origin bappy 2>&1 | tail -2
echo "--- branches that exist on the remote ---"
git ls-remote --heads origin

To /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch08_10_git/git_sandbox/github-learning-t

est.git
 * [new branch]      bappy -> bappy


--- branches that exist on the remote ---


05de748b99cecb8b58f51d92251d95c113925e60	refs/heads/bappy
2491c7b796e9d393f0ea15c92a409e7134d748c3	r

efs/heads/main


## 10.3 · Switch back: `main` is unchanged  (video 02:24)

In [17]:
%%bash
cd dev1
git checkout -q main
echo "on $(git branch --show-current): test3.py has $(wc -l < test3.py | tr -d ' ') lines"
git checkout -q bappy
echo "on $(git branch --show-current): test3.py has $(wc -l < test3.py | tr -d ' ') lines"
cat test3.py
git checkout -q main

on main: test3.py has 0 lines


on bappy: test3.py has 2 lines


print("hello world")
print("this is new")


## 10.4 · Merge the branch (beyond the video: the step the video leaves out)
On GitHub you'd open a **pull request** (bappy → main) and click *Merge*. Locally, it looks like this:

In [18]:
%%bash
cd dev1
git merge bappy 2>&1 | tail -3
git push -q origin main
git branch -d bappy
git push -q origin --delete bappy && echo "branch deleted locally and on the remote"
cat test3.py

Fast-forward
 test3.py | 2 ++
 1 file changed, 2 insertions(+)


Deleted branch bappy (was 05de748).


branch deleted locally and on the remote


print("hello world")
print("this is new")


## 10.5 · A merge conflict, and how to resolve it (beyond the video)
Two branches change **the same line** of a file differently.

In [19]:
%%bash
cd dev1
git checkout -q -b feature/greeting
printf 'print("hello from the feature branch")\nprint("this is new")\n' > test3.py
git commit -q -am "feature: new greeting"
git checkout -q main
printf 'print("hello from main")\nprint("this is new")\n' > test3.py
git commit -q -am "main: different greeting"
git merge feature/greeting 2>&1 || true
echo "===== test3.py with conflict markers ====="
cat test3.py

Auto-merging test3.py
CONFLICT (content): Merge conflict in test3.py


Automatic merge failed; fix conflicts and then commit the result.
===== test3.py with conflict marke

rs =====


<<<<<<< HEAD
print("hello from main")
print("hello from the feature branch")
>>>>>>> feature

/greeting
print("this is new")


In [20]:
%%bash
cd dev1
# resolve: write the version we want (keep both greetings) and remove the markers
printf 'print("hello from main")\nprint("hello from the feature branch")\nprint("this is new")\n' > test3.py
git add test3.py
git commit -q --no-edit
git push -q origin main
echo "resolved:"; cat test3.py

resolved:


print("hello from main")
print("hello from the feature branch")
print("this is new")


## 10.6 · The whole history as a graph

In [21]:
%%bash
cd dev1
git log --oneline --graph --all

*   6d7848a Merge branch 'feature/greeting'
|\  


| * 6970fd3 feature: new greeting


* | e834df2 main: different greeting
|/  
* 05de748 new code added
* 2491c7b files added
* e8c53e0 R

evert "buggy change"
* 3b72f8d buggy change
*   530417d Merge branch 'main' of /Users/hemanthreddy/D

esktop/data_science/MLOps/projects/ch08_10_git/git_sandbox/github-learning-test
|\  
| * b416f04 add

 utils
* | d32222e add notes
|/  
* 51051d1 test.py file updated
* 77484a4 test.py file added
* c7b0

401 Initial commit


## Summary

| Chapter | Commands practised |
|---|---|
| 08 | `git init --bare` (stands in for GitHub), `git clone`, `.git/`, `git remote -v`, `.gitignore` + `git check-ignore` |
| 09 | `git status`, `git add <file>` / `git add .`, `git commit -m`, `git push origin main`, `git pull`, rejected push, `git log`, `git show`, `git restore --source`, `git revert` |
| 10 | `git branch`, `git checkout -b`, `git push origin <branch>`, `git checkout main`, `git merge`, resolving a conflict, `git branch -d`, `git log --graph` |

Delete `git_sandbox/` whenever you like; re-running the notebook rebuilds it.